# Redshift vs maximum detected time (per band)

Reads `lanl_extinguished_photometry.parquet` and, for each Roman band, computes per `(simulation, angle, redshift)` the latest `time_days` at which `mag_ab_<band> <= mag_lim` (= 28). Shows the redshift–vs–`t_max_detected` distribution as a hexbin density per band, restricted to `z ∈ [0.003, 0.5]`.

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

PARQUET = 'lanl_extinguished_photometry.parquet'
Z_MIN, Z_MAX = 0.003, 0.5
MAG_LIM = 28.0

# Roman bands in order, with column name -> short label.
BANDS = [
    ('mag_ab_R062-R', 'R062'),
    ('mag_ab_Z087-Z', 'Z087'),
    ('mag_ab_Y106-Y', 'Y106'),
    ('mag_ab_J129-J', 'J129'),
    ('mag_ab_H158-H', 'H158'),
    ('mag_ab_F184-F', 'F184'),
]

In [ ]:
# Sanity preview: pick a kilonova light curve near a target redshift and plot it
# per band. Uses row-group statistics to read only one candidate row group, then
# picks the (sim, angle, z) whose z is closest to Z_TARGET.
import random

Z_TARGET = 0.1

pf_preview = pq.ParquetFile(PARQUET)
md = pf_preview.metadata
z_col_idx = pf_preview.schema_arrow.names.index('redshift')

candidates = []
for rg in range(pf_preview.num_row_groups):
    stats = md.row_group(rg).column(z_col_idx).statistics
    if stats is None:
        continue
    if stats.min <= Z_TARGET <= stats.max:
        candidates.append(rg)
print(f'row groups containing z={Z_TARGET}: {len(candidates)}/{pf_preview.num_row_groups}')

rg = random.choice(candidates)
cols = ['simulation_id', 'angle_index', 'redshift', 'time_days'] + [c for c, _ in BANDS]
df_rg = pf_preview.read_row_group(rg, columns=cols).to_pandas()

# Pick the (sim, angle, z) key whose redshift is closest to the target.
keys_in_rg = (df_rg[['simulation_id', 'angle_index', 'redshift']]
                .drop_duplicates()
                .reset_index(drop=True))
idx = (keys_in_rg['redshift'] - Z_TARGET).abs().idxmin()
sim_id = int(keys_in_rg.loc[idx, 'simulation_id'])
ang_id = int(keys_in_rg.loc[idx, 'angle_index'])
z_chosen = float(keys_in_rg.loc[idx, 'redshift'])

lc = (df_rg[(df_rg['simulation_id'] == sim_id)
            & (df_rg['angle_index'] == ang_id)
            & (df_rg['redshift'] == z_chosen)]
        .sort_values('time_days'))
print(f'picked row group {rg}, simulation_id={sim_id}, angle_index={ang_id}, '
      f'z={z_chosen:.4f}, n_points={len(lc)}, '
      f't_days=[{lc["time_days"].min():.3f}, {lc["time_days"].max():.3f}]')

band_colors = plt.cm.turbo(np.linspace(0.05, 0.95, len(BANDS)))

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, sharey=True)
axes = axes.ravel()
for ax, (col, label), bcolor in zip(axes, BANDS, band_colors):
    ax.plot(lc['time_days'], lc[col], color=bcolor, lw=1.6, marker='.', ms=3)
    ax.axhline(MAG_LIM, color='k', ls='--', lw=0.8, alpha=0.6,
               label=f'mag$_{{lim}}$={MAG_LIM}')
    ax.set_title(label, color=bcolor, fontweight='bold')
    ax.grid(alpha=0.2)
    ax.legend(loc='lower right', fontsize=8)

axes[0].invert_yaxis()

for ax in axes[-3:]:
    ax.set_xlabel('time [days]')
for ax in axes[::3]:
    ax.set_ylabel('mag (AB)')

fig.suptitle(f'Kilonova light curve near z={Z_TARGET} '
             f'(sim={sim_id}, angle={ang_id}, z={z_chosen:.4f})', y=1.0)
fig.tight_layout()
plt.show()

del df_rg, lc

In [ ]:
# Stream the parquet row group by row group: 366M rows / 17 GB don't fit in RAM,
# but each row group is ~251k rows and contains whole light curves (no LC is split
# across row groups), so we can reduce to t_max per (sim, angle, z) on the fly.
pf = pq.ParquetFile(PARQUET)
print(f'row groups: {pf.num_row_groups}, total rows: {pf.metadata.num_rows:,}')

KEYS = ['simulation_id', 'angle_index', 'redshift']
NEEDED = KEYS + ['time_days'] + [c for c, _ in BANDS]

partials = {label: [] for _, label in BANDS}

for rg in range(pf.num_row_groups):
    df = pf.read_row_group(rg, columns=NEEDED).to_pandas()
    df = df[(df['redshift'] >= Z_MIN) & (df['redshift'] <= Z_MAX)]
    if df.empty:
        continue
    for col, label in BANDS:
        mask = df[col].to_numpy() <= MAG_LIM
        if not mask.any():
            continue
        sub = df.loc[mask, KEYS + ['time_days']]
        partials[label].append(
            sub.groupby(KEYS, sort=False)['time_days'].max().reset_index()
        )
    if rg % 50 == 0:
        print(f'  row group {rg}/{pf.num_row_groups}')
print('done streaming')

In [ ]:
# Final reduction across row-group partials (defensive: a single groupby-max).
tmax_per_band = {}
for _, label in BANDS:
    if not partials[label]:
        tmax_per_band[label] = pd.DataFrame(columns=KEYS + ['t_max_detected'])
        print(f'{label}: 0 light curves with >=1 detection')
        continue
    cat = pd.concat(partials[label], ignore_index=True)
    tmax = (cat.groupby(KEYS, sort=False)['time_days']
                .max()
                .reset_index()
                .rename(columns={'time_days': 't_max_detected'}))
    tmax_per_band[label] = tmax
    print(f'{label}: {len(tmax):,} light curves with >=1 detection')

In [ ]:
# Which simulation_ids are still detectable at z > 0.3 in at least one band?
# Uses tmax_per_band: rows there mean the light curve reached mag <= MAG_LIM at
# the corresponding (sim, angle, z).
Z_CUT = 0.3

detectable_sims = set()
per_band_sims = {}
for _, label in BANDS:
    t = tmax_per_band[label]
    sims = set(t.loc[t['redshift'] > Z_CUT, 'simulation_id'].unique())
    per_band_sims[label] = sims
    detectable_sims |= sims

print(f'simulation_ids detectable at z > {Z_CUT} (any band): '
      f'{len(detectable_sims)} -> {sorted(detectable_sims)}')
print()
for _, label in BANDS:
    sims = per_band_sims[label]
    print(f'  {label}: {len(sims):3d} sims  ->  {sorted(sims)}')

In [ ]:
# Violin plot of t_max_detected per redshift bin, one panel per Roman band.
# Bins log-spaced in z, color per band from the Extinction.ipynb palette.
band_colors = plt.cm.turbo(np.linspace(0.05, 0.95, len(BANDS)))

N_BINS = 12
z_edges = np.geomspace(Z_MIN, Z_MAX, N_BINS + 1)
z_centers = np.sqrt(z_edges[:-1] * z_edges[1:])  # geometric mean
bin_width = np.diff(z_edges)
bin_labels = [f'{lo:.3f}–{hi:.3f}' for lo, hi in zip(z_edges[:-1], z_edges[1:])]

t_max_global = max(t['t_max_detected'].max() for t in tmax_per_band.values())

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, (_, label), bcolor in zip(axes, BANDS, band_colors):
    t = tmax_per_band[label]
    z = t['redshift'].to_numpy()
    y = t['t_max_detected'].to_numpy()
    bin_idx = np.digitize(z, z_edges) - 1  # 0..N_BINS-1

    data, positions, widths, counts = [], [], [], []
    for i in range(N_BINS):
        sel = (bin_idx == i) & (y > 0)
        if sel.sum() < 5:  # skip near-empty bins; violins need spread
            continue
        data.append(y[sel])
        positions.append(z_centers[i])
        widths.append(bin_width[i] * 0.85)
        counts.append(sel.sum())

    if data:
        parts = ax.violinplot(data, positions=positions, widths=widths,
                              showmedians=True, showextrema=False)
        for body in parts['bodies']:
            body.set_facecolor(bcolor)
            body.set_edgecolor(bcolor)
            body.set_alpha(0.55)
        if 'cmedians' in parts:
            parts['cmedians'].set_color(bcolor)
            parts['cmedians'].set_linewidth(1.6)

        # Count per bin annotated along the top.
        for pos, n in zip(positions, counts):
            ax.text(pos, t_max_global * 0.97, f'{n:,}',
                    ha='center', va='top', fontsize=7, color=bcolor, alpha=0.9)

    ax.set_xscale('log')
    ax.set_xlim(Z_MIN, Z_MAX)
    ax.set_ylim(0, t_max_global * 1.02)
    ax.set_title(f'{label}  (n={len(t):,})', color=bcolor, fontweight='bold')
    ax.grid(alpha=0.2, which='both')

for ax in axes[-3:]:
    ax.set_xlabel('redshift (log bins)')
for ax in axes[::3]:
    ax.set_ylabel('max detected time [days]')

fig.suptitle(f'Distribution of latest detected epoch per redshift bin '
             f'(mag$_\\mathrm{{lim}}$ = {MAG_LIM})', y=1.0)
fig.tight_layout()
fig.savefig('redshift_vs_detected_tmax.png', dpi=150, bbox_inches='tight')
plt.show()